# Setup & Analisis Korelasi Grup Housing

In [ ]:
import sys
import pandas as pd
from itertools import combinations
sys.path.append("../src")

from features import (
    load_cleaned_dataset,
    build_housing_groups,
    compute_housing_variant_correlation,
    compute_variant_target_correlation,
    select_housing_representative,
    compute_numeric_correlation_matrices,
    find_redundant_numeric_pairs,
    drop_redundant_numeric_features,
    cramers_v,
    compute_categorical_association_matrix,
    evaluate_variance_threshold,
    compute_near_zero_variance_target_correlation,
    drop_near_zero_variance_features,
    create_credit_goods_features,
    build_final_feature_lists,
    FINANCIAL_LOG_PAIRS,
    save_feature_engineered_dataset,
    save_feature_engineering_metadata,
)

df = load_cleaned_dataset("../data/processed/application_train_clean.csv")
print(f"Shape data cleaned: {df.shape}")

housing_groups = build_housing_groups()
print("\nGrup housing yang terdeteksi:")
for base, cols in housing_groups.items():
    print(f"  {base}: {cols}")

intra_corr = compute_housing_variant_correlation(df, housing_groups) # Intra-group Correlation 
for base, corr_matrix in intra_corr.items():                            # Correlation Matrix
    print(f"\nKorelasi intra-grup {base}:")
    print(corr_matrix)

target_corr = compute_variant_target_correlation(df, housing_groups)
print("\nKorelasi tiap varian terhadap TARGET:")
print(target_corr)

Shape data cleaned: (307511, 101)

Grup housing yang terdeteksi:
  FLOORSMAX: ['FLOORSMAX_AVG', 'FLOORSMAX_MEDI', 'FLOORSMAX_MODE']
  LIVINGAREA: ['LIVINGAREA_AVG', 'LIVINGAREA_MEDI', 'LIVINGAREA_MODE']
  APARTMENTS: ['APARTMENTS_AVG', 'APARTMENTS_MEDI', 'APARTMENTS_MODE']
  YEARS_BEGINEXPLUATATION: ['YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BEGINEXPLUATATION_MODE']
  ENTRANCES: ['ENTRANCES_AVG', 'ENTRANCES_MEDI', 'ENTRANCES_MODE']

Korelasi intra-grup FLOORSMAX:
                FLOORSMAX_AVG  FLOORSMAX_MEDI  FLOORSMAX_MODE
FLOORSMAX_AVG          1.0000          0.9973          0.9866
FLOORSMAX_MEDI         0.9973          1.0000          0.9890
FLOORSMAX_MODE         0.9866          0.9890          1.0000

Korelasi intra-grup LIVINGAREA:
                 LIVINGAREA_AVG  LIVINGAREA_MEDI  LIVINGAREA_MODE
LIVINGAREA_AVG           1.0000           0.9958           0.9732
LIVINGAREA_MEDI          0.9958           1.0000           0.9758
LIVINGAREA_MODE          

# Seleksi Fitur Housing & Cek Redundancy Numerik

In [2]:
df, dropped_housing_cols = select_housing_representative(df)
print(f"Kolom yang dibuang: {dropped_housing_cols}")
print(f"Shape setelah seleksi housing: {df.shape}")

corr_matrices = compute_numeric_correlation_matrices(df)
redundant_pairs = find_redundant_numeric_pairs(corr_matrices)
print("\nPasangan numerik redundan (|r| > 0.9):")
print(redundant_pairs)

Kolom yang dibuang: ['FLOORSMAX_AVG', 'FLOORSMAX_MODE', 'LIVINGAREA_AVG', 'LIVINGAREA_MODE', 'APARTMENTS_AVG', 'APARTMENTS_MODE', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BEGINEXPLUATATION_MODE', 'ENTRANCES_AVG', 'ENTRANCES_MODE']
Shape setelah seleksi housing: (307511, 91)

Pasangan numerik redundan (|r| > 0.9):
                      feature_1                 feature_2  pearson_corr  \
0                    AMT_CREDIT            LOG_AMT_CREDIT        0.9198   
1               AMT_GOODS_PRICE       LOG_AMT_GOODS_PRICE        0.9168   
2              AMT_INCOME_TOTAL      LOG_AMT_INCOME_TOTAL        0.3987   
3                   AMT_ANNUITY           LOG_AMT_ANNUITY        0.9306   
4                FLAG_EMP_PHONE    DAYS_EMPLOYED_SENTINEL       -0.9999   
5      OBS_60_CNT_SOCIAL_CIRCLE  OBS_30_CNT_SOCIAL_CIRCLE        0.9985   
6                LOG_AMT_CREDIT       LOG_AMT_GOODS_PRICE        0.9883   
7               AMT_GOODS_PRICE                AMT_CREDIT        0.9867   
8            

# TODO — Kandidat Fitur Turunan (Feature Creation, belum dieksekusi)
Insight dari redundancy check: AMT_CREDIT dan AMT_GOODS_PRICE dipertahankan
keduanya karena beda makna bisnis (jumlah pinjaman vs harga barang). Selisih
dan rasio keduanya berpotensi jadi fitur baru yang menangkap "pinjaman
tambahan di luar harga barang":
- AMT_CREDIT_MINUS_GOODS_PRICE = AMT_CREDIT - AMT_GOODS_PRICE
- AMT_CREDIT_TO_GOODS_RATIO = AMT_CREDIT / AMT_GOODS_PRICE

Akan dieksekusi di bagian Feature Creation, setelah redundancy check dan
VarianceThreshold selesai.

In [3]:
crosstab_result = pd.crosstab(df["FLAG_EMP_PHONE"], df["DAYS_EMPLOYED_SENTINEL"])
print("Crosstab FLAG_EMP_PHONE vs DAYS_EMPLOYED_SENTINEL:")
print(crosstab_result)

n_consistent_inverse = (
    (df["FLAG_EMP_PHONE"] == 1) & (df["DAYS_EMPLOYED_SENTINEL"] == 0)
) | (
    (df["FLAG_EMP_PHONE"] == 0) & (df["DAYS_EMPLOYED_SENTINEL"] == 1)
)
pct_consistent = n_consistent_inverse.mean() * 100
print(f"\n{pct_consistent:.4f}% baris mengikuti pola inverse sempurna (FLAG_EMP_PHONE=1 <-> SENTINEL=0)")
print(f"Sisa baris ({100 - pct_consistent:.4f}%) adalah pengecualian dari pola tersebut")

Crosstab FLAG_EMP_PHONE vs DAYS_EMPLOYED_SENTINEL:
DAYS_EMPLOYED_SENTINEL       0      1
FLAG_EMP_PHONE                       
0                           12  55374
1                       252125      0

99.9961% baris mengikuti pola inverse sempurna (FLAG_EMP_PHONE=1 <-> SENTINEL=0)
Sisa baris (0.0039%) adalah pengecualian dari pola tersebut


# Drop Fitur Numerik Redundan (Explicit)
Tujuan kode ini: memanggil drop_redundant_numeric_features() yang baru ditambahkan ke features.py, untuk benar-benar membuang 3 kolom yang sudah disepakati (FLAG_EMP_PHONE, OBS_60_CNT_SOCIAL_CIRCLE, REGION_RATING_CLIENT) dari df. Setelah ini, shape harus berubah dari (307511, 91) menjadi (307511, 88). Ada assert untuk memastikan ketiga kolom benar-benar hilang, supaya kalau ada kesalahan (misal fungsi belum tersimpan dengan benar), error langsung ketahuan di sini, bukan menyebar ke langkah berikutnya.

In [4]:
from features import drop_redundant_numeric_features

df, dropped_numeric_cols = drop_redundant_numeric_features(df)

print(f"Kolom yang dibuang: {dropped_numeric_cols}")
print(f"Shape setelah drop redundant numeric features: {df.shape}")
assert all(col not in df.columns for col in dropped_numeric_cols), "Masih ada kolom yang seharusnya dibuang!"

Kolom yang dibuang: ['FLAG_EMP_PHONE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'REGION_RATING_CLIENT']
Shape setelah drop redundant numeric features: (307511, 88)


Catatan: Singkatnya: korelasi tinggi **bukan otomatis berarti buang kolom** — itu cuma sinyal untuk dicek maknanya dulu. Kalau ada 1 pasangan redundan, cuma **1 kolom yang dibuang** (bukan dua-duanya), makanya dari 6 kolom di 3 pasangan itu cuma 3 yang hilang, sisanya dipertahankan sebagai wakil. `FLAG_EMP_PHONE`, `OBS_60_CNT_SOCIAL_CIRCLE`, dan `REGION_RATING_CLIENT` dibuang karena memang duplikat/kebetulan mirip dengan pasangannya. Sementara `AMT_CREDIT` vs `AMT_GOODS_PRICE` **sengaja dipertahankan** meski korelasinya tinggi, karena maknanya beda (pinjaman vs harga barang) dan selisih/rasionya justru berpotensi jadi fitur baru yang berguna. Jadi keputusannya selalu berdasarkan **konteks bisnis**, bukan cuma angka korelasi semata.

# Identifikasi Kolom Kategorikal
Tujuan kode ini: sebelum menjalankan Cramér's V, kita harus tahu dulu kolom mana saja yang benar-benar bertipe kategorikal di df saat ini. Kita tidak asumsikan daftar kolom dari awal (misal RARE_CATEGORY_COLS di cleaning.py), karena tujuan di sini berbeda: kita mau cek seluruh kolom kategorikal yang tersisa di dataframe, termasuk yang mungkin belum pernah direview redundancy-nya (contoh: WALLSMATERIAL_MODE, HOUSETYPE_MODE, NAME_CONTRACT_TYPE, dll).

In [5]:
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"Jumlah kolom kategorikal terdeteksi: {len(categorical_cols)}")
print(categorical_cols)

Jumlah kolom kategorikal terdeteksi: 15
['OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'CODE_GENDER', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE', 'HOUSETYPE_MODE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'NAME_TYPE_SUITE', 'WEEKDAY_APPR_PROCESS_START', 'FLAG_OWN_REALTY']


# Hitung Matriks Cramér's V
Tujuan kode ini: memanggil compute_categorical_association_matrix() untuk menghitung kekuatan asosiasi antar semua pasangan kolom kategorikal, lalu memfilter pasangan mana saja yang punya Cramér's V di atas threshold 0.9 (analog dengan REDUNDANCY_THRESHOLD yang dipakai di redundancy numerik). Filtering pasangan ini ditulis langsung di notebook (bukan di features.py) karena sifatnya eksploratif satu kali untuk tahap ini, bukan logika reusable yang dipanggil berulang di pipeline.

In [6]:
from features import compute_categorical_association_matrix

cramers_matrix = compute_categorical_association_matrix(df, categorical_cols)
print("Matriks Cramér's V:")
print(cramers_matrix)

high_association_pairs = []
for col_a, col_b in combinations(categorical_cols, 2):
    v = cramers_matrix.loc[col_a, col_b]
    if v > 0.9:
        high_association_pairs.append({"feature_1": col_a, "feature_2": col_b, "cramers_v": v})

high_association_df = pd.DataFrame(high_association_pairs)
print("\nPasangan kategorikal dengan Cramér's V > 0.9:")
print(high_association_df)

Matriks Cramér's V:
                            OCCUPATION_TYPE  ORGANIZATION_TYPE  \
OCCUPATION_TYPE                      1.0000             0.3232   
ORGANIZATION_TYPE                    0.3232             1.0000   
NAME_INCOME_TYPE                     0.3825             0.5587   
NAME_EDUCATION_TYPE                  0.1874             0.1280   
CODE_GENDER                          0.5077             0.3371   
WALLSMATERIAL_MODE                   0.0324             0.0471   
EMERGENCYSTATE_MODE                  0.0568             0.0840   
HOUSETYPE_MODE                       0.0461             0.0671   
NAME_FAMILY_STATUS                   0.1020             0.1337   
NAME_HOUSING_TYPE                    0.0441             0.0798   
NAME_CONTRACT_TYPE                   0.0612             0.0665   
FLAG_OWN_CAR                         0.2565             0.2020   
NAME_TYPE_SUITE                      0.0224             0.0264   
WEEKDAY_APPR_PROCESS_START           0.0180             

#  Evaluasi VarianceThreshold
Tujuan kode ini: memanggil evaluate_variance_threshold() untuk mengukur variance aktual dari 18 fitur near-zero variance (FLAG_DOCUMENT_*, FLAG_MOBIL, FLAG_CONT_MOBILE) secara kuantitatif, supaya keputusan buang/pertahankan nanti berbasis angka bukan asumsi.

In [7]:
from features import evaluate_variance_threshold

variance_report = evaluate_variance_threshold(df)
print("Evaluasi VarianceThreshold untuk fitur near-zero variance:")
print(variance_report)

Evaluasi VarianceThreshold untuk fitur near-zero variance:
             feature  variance  below_threshold
0         FLAG_MOBIL  0.000003             True
1   FLAG_DOCUMENT_12  0.000007             True
2   FLAG_DOCUMENT_10  0.000023             True
3    FLAG_DOCUMENT_2  0.000042             True
4    FLAG_DOCUMENT_4  0.000081             True
5    FLAG_DOCUMENT_7  0.000192             True
6   FLAG_DOCUMENT_17  0.000267             True
7   FLAG_DOCUMENT_21  0.000335             True
8   FLAG_DOCUMENT_20  0.000507             True
9   FLAG_DOCUMENT_19  0.000595             True
10  FLAG_DOCUMENT_15  0.001208             True
11  FLAG_CONT_MOBILE  0.001863             True
12  FLAG_DOCUMENT_14  0.002928             True
13  FLAG_DOCUMENT_13  0.003513             True
14   FLAG_DOCUMENT_9  0.003881             True
15  FLAG_DOCUMENT_11  0.003897             True
16  FLAG_DOCUMENT_18  0.008064             True
17  FLAG_DOCUMENT_16  0.009830             True


# Korelasi Fitur Near-Zero Variance terhadap TARGET

Tujuan kode ini: memanggil fungsi baru di atas, lalu menggabungkan (merge) hasilnya dengan variance_report yang sudah ada, supaya kedua bukti (variance dan korelasi target) bisa dilihat berdampingan dalam satu tabel — memudahkan kita mengambil keputusan akhir per kolom, bukan cuma lihat satu angka saja.

In [8]:
from features import compute_near_zero_variance_target_correlation

target_corr_nzv = compute_near_zero_variance_target_correlation(df)

variance_target_review = variance_report.merge(target_corr_nzv, on="feature")
variance_target_review = variance_target_review.sort_values("corr_with_target", key=abs, ascending=False).reset_index(drop=True)

print("Review gabungan variance & korelasi TARGET untuk fitur near-zero variance:")
print(variance_target_review)

Review gabungan variance & korelasi TARGET untuk fitur near-zero variance:
             feature  variance  below_threshold  corr_with_target
0   FLAG_DOCUMENT_13  0.003513             True           -0.0116
1   FLAG_DOCUMENT_16  0.009830             True           -0.0116
2   FLAG_DOCUMENT_14  0.002928             True           -0.0095
3   FLAG_DOCUMENT_18  0.008064             True           -0.0080
4   FLAG_DOCUMENT_15  0.001208             True           -0.0065
5    FLAG_DOCUMENT_2  0.000042             True            0.0054
6    FLAG_DOCUMENT_9  0.003881             True           -0.0044
7   FLAG_DOCUMENT_11  0.003897             True           -0.0042
8   FLAG_DOCUMENT_21  0.000335             True            0.0037
9   FLAG_DOCUMENT_17  0.000267             True           -0.0034
10   FLAG_DOCUMENT_4  0.000081             True           -0.0027
11   FLAG_DOCUMENT_7  0.000192             True           -0.0015
12  FLAG_DOCUMENT_10  0.000023             True           -0.0014
1

# Drop Fitur Near-Zero Variance

Tujuan kode ini: memanggil fungsi di atas, menampilkan daftar kolom yang dibuang dan shape sebelum/sesudah, plus assert untuk memastikan seluruh 18 kolom benar-benar hilang dari df.

In [9]:
from features import drop_near_zero_variance_features

df, dropped_nzv_cols = drop_near_zero_variance_features(df)

print(f"Jumlah kolom yang dibuang: {len(dropped_nzv_cols)}")
print(f"Kolom yang dibuang: {dropped_nzv_cols}")
print(f"Shape setelah drop near-zero variance features: {df.shape}")
assert all(col not in df.columns for col in dropped_nzv_cols), "Masih ada kolom yang seharusnya dibuang!"

Jumlah kolom yang dibuang: 18
Kolom yang dibuang: ['FLAG_MOBIL', 'FLAG_CONT_MOBILE', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21']
Shape setelah drop near-zero variance features: (307511, 70)


# Cek Distribusi AMT_GOODS_PRICE
Tujuan kode ini: murni eksploratif satu kali (bukan logika reusable, jadi ditulis langsung di notebook, bukan di features.py), untuk memverifikasi apakah ada baris dengan AMT_GOODS_PRICE = 0 atau nilai negatif, sebelum kita memutuskan strategi penanganan pembagian dengan nol untuk AMT_CREDIT_TO_GOODS_RATIO.

In [10]:
n_zero_goods_price = (df["AMT_GOODS_PRICE"] == 0).sum()
n_negative_goods_price = (df["AMT_GOODS_PRICE"] < 0).sum()
n_null_goods_price = df["AMT_GOODS_PRICE"].isnull().sum()

print(f"Jumlah baris AMT_GOODS_PRICE == 0: {n_zero_goods_price}")
print(f"Jumlah baris AMT_GOODS_PRICE < 0: {n_negative_goods_price}")
print(f"Jumlah baris AMT_GOODS_PRICE null: {n_null_goods_price}")
print(f"\nStatistik deskriptif AMT_GOODS_PRICE:")
print(df["AMT_GOODS_PRICE"].describe())

n_zero_credit = (df["AMT_CREDIT"] == 0).sum()
print(f"\nJumlah baris AMT_CREDIT == 0 (untuk kelengkapan, bukan pembagi): {n_zero_credit}")

Jumlah baris AMT_GOODS_PRICE == 0: 0
Jumlah baris AMT_GOODS_PRICE < 0: 0
Jumlah baris AMT_GOODS_PRICE null: 0

Statistik deskriptif AMT_GOODS_PRICE:
count    3.075110e+05
mean     5.383163e+05
std      3.692890e+05
min      4.050000e+04
25%      2.385000e+05
50%      4.500000e+05
75%      6.795000e+05
max      4.050000e+06
Name: AMT_GOODS_PRICE, dtype: float64

Jumlah baris AMT_CREDIT == 0 (untuk kelengkapan, bukan pembagi): 0


# Feature Creation - Selisih & Rasio AMT_CREDIT vs AMT_GOODS_PRICE

Tujuan kode ini: memanggil fungsi di atas, lalu menampilkan statistik deskriptif dan korelasi kedua fitur baru terhadap TARGET, supaya kita bisa langsung menilai apakah fitur ini benar-benar informatif sebelum masuk ke daftar fitur final.

In [11]:
from features import create_credit_goods_features
import numpy as np

df = create_credit_goods_features(df)

print(f"Shape setelah feature creation: {df.shape}")
print("\nStatistik AMT_CREDIT_MINUS_GOODS_PRICE:")
print(df["AMT_CREDIT_MINUS_GOODS_PRICE"].describe())
print("\nStatistik AMT_CREDIT_TO_GOODS_RATIO:")
print(df["AMT_CREDIT_TO_GOODS_RATIO"].describe())

n_null_ratio = df["AMT_CREDIT_TO_GOODS_RATIO"].isnull().sum()
n_inf_ratio = np.isinf(df["AMT_CREDIT_TO_GOODS_RATIO"]).sum()
print(f"\nJumlah null di AMT_CREDIT_TO_GOODS_RATIO: {n_null_ratio}")
print(f"Jumlah inf di AMT_CREDIT_TO_GOODS_RATIO: {n_inf_ratio}")

new_feature_corr = df[["AMT_CREDIT_MINUS_GOODS_PRICE", "AMT_CREDIT_TO_GOODS_RATIO", "TARGET"]].corr()["TARGET"]
print("\nKorelasi fitur baru terhadap TARGET:")
print(new_feature_corr)

Shape setelah feature creation: (307511, 73)

Statistik AMT_CREDIT_MINUS_GOODS_PRICE:
count    307511.000000
mean      60709.705339
std       71034.266632
min     -765000.000000
25%           0.000000
50%       39204.000000
75%       99792.000000
max      900000.000000
Name: AMT_CREDIT_MINUS_GOODS_PRICE, dtype: float64

Statistik AMT_CREDIT_TO_GOODS_RATIO:
count    307511.000000
mean          1.122542
std           0.125542
min           0.150000
25%           1.000000
50%           1.118800
75%           1.198000
max           6.000000
Name: AMT_CREDIT_TO_GOODS_RATIO, dtype: float64

Jumlah null di AMT_CREDIT_TO_GOODS_RATIO: 0
Jumlah inf di AMT_CREDIT_TO_GOODS_RATIO: 0

Korelasi fitur baru terhadap TARGET:
AMT_CREDIT_MINUS_GOODS_PRICE    0.033914
AMT_CREDIT_TO_GOODS_RATIO       0.068474
TARGET                          1.000000
Name: TARGET, dtype: float64


#  Tambah LOG_AMT_CREDIT_TO_GOODS_RATIO

Tujuan kode ini: karena create_credit_goods_features() sudah pernah dipanggil sebelumnya di notebook Anda (menghasilkan df dengan 72 kolom, tanpa LOG_RATIO), cell ini harus memuat ulang dari checkpoint sebelum feature creation agar tidak terjadi duplikasi kolom saat fungsi dipanggil dua kali pada df yang sama.

In [12]:
from features import create_credit_goods_features

df = create_credit_goods_features(df)

print(f"Shape setelah revisi feature creation: {df.shape}")
print("\nStatistik LOG_AMT_CREDIT_TO_GOODS_RATIO:")
print(df["LOG_AMT_CREDIT_TO_GOODS_RATIO"].describe())

n_null_log_ratio = df["LOG_AMT_CREDIT_TO_GOODS_RATIO"].isnull().sum()
n_inf_log_ratio = np.isinf(df["LOG_AMT_CREDIT_TO_GOODS_RATIO"]).sum()
print(f"\nJumlah null di LOG_AMT_CREDIT_TO_GOODS_RATIO: {n_null_log_ratio}")
print(f"Jumlah inf di LOG_AMT_CREDIT_TO_GOODS_RATIO: {n_inf_log_ratio}")

log_ratio_corr = df[["LOG_AMT_CREDIT_TO_GOODS_RATIO", "TARGET"]].corr().loc["LOG_AMT_CREDIT_TO_GOODS_RATIO", "TARGET"]
print(f"\nKorelasi LOG_AMT_CREDIT_TO_GOODS_RATIO terhadap TARGET: {log_ratio_corr:.4f}")

Shape setelah revisi feature creation: (307511, 73)

Statistik LOG_AMT_CREDIT_TO_GOODS_RATIO:
count    307511.000000
mean          0.109570
std           0.109616
min          -1.897120
25%           0.000000
50%           0.112257
75%           0.180653
max           1.791759
Name: LOG_AMT_CREDIT_TO_GOODS_RATIO, dtype: float64

Jumlah null di LOG_AMT_CREDIT_TO_GOODS_RATIO: 0
Jumlah inf di LOG_AMT_CREDIT_TO_GOODS_RATIO: 0

Korelasi LOG_AMT_CREDIT_TO_GOODS_RATIO terhadap TARGET: 0.0668


# Final Feature List Split

Tujuan kode ini: memanggil fungsi di atas, lalu menampilkan jumlah fitur di masing-masing jalur dan memverifikasi bahwa pasangan raw-LOG benar-benar terpisah dengan benar (tidak ada raw dan LOG yang sama-sama masuk satu jalur).



In [13]:
from features import build_final_feature_lists, FINANCIAL_LOG_PAIRS

feature_lists = build_final_feature_lists(df)
tree_features = feature_lists["tree"]
linear_mlp_features = feature_lists["linear_mlp"]

print(f"Jumlah fitur jalur tree-based: {len(tree_features)}")
print(f"Jumlah fitur jalur linear/MLP: {len(linear_mlp_features)}")

for raw_col, log_col in FINANCIAL_LOG_PAIRS.items():
    print(f"\n{raw_col}: tree={raw_col in tree_features}, linear_mlp={raw_col in linear_mlp_features}")
    print(f"{log_col}: tree={log_col in tree_features}, linear_mlp={log_col in linear_mlp_features}")

print(f"\nAMT_CREDIT_MINUS_GOODS_PRICE: tree={'AMT_CREDIT_MINUS_GOODS_PRICE' in tree_features}, linear_mlp={'AMT_CREDIT_MINUS_GOODS_PRICE' in linear_mlp_features}")

Jumlah fitur jalur tree-based: 66
Jumlah fitur jalur linear/MLP: 66

AMT_INCOME_TOTAL: tree=True, linear_mlp=False
LOG_AMT_INCOME_TOTAL: tree=False, linear_mlp=True

AMT_CREDIT: tree=True, linear_mlp=False
LOG_AMT_CREDIT: tree=False, linear_mlp=True

AMT_ANNUITY: tree=True, linear_mlp=False
LOG_AMT_ANNUITY: tree=False, linear_mlp=True

AMT_GOODS_PRICE: tree=True, linear_mlp=False
LOG_AMT_GOODS_PRICE: tree=False, linear_mlp=True

AMT_CREDIT_TO_GOODS_RATIO: tree=True, linear_mlp=False
LOG_AMT_CREDIT_TO_GOODS_RATIO: tree=False, linear_mlp=True

AMT_CREDIT_MINUS_GOODS_PRICE: tree=True, linear_mlp=True


# 12. Simpan Dataset & Metadata Feature Engineering Final

Tujuan kode ini: memanggil kedua fungsi di atas menggunakan variabel-variabel yang sudah ada dari cell-cell sebelumnya (dropped_housing_cols, dropped_numeric_cols, dropped_nzv_cols, feature_lists), lalu menyimpan kedua artifact ke data/processed/.


In [14]:
from features import save_feature_engineered_dataset, save_feature_engineering_metadata

save_feature_engineered_dataset(df, "../data/processed/application_train_featured.csv")

save_feature_engineering_metadata(
    dropped_housing_cols=dropped_housing_cols,
    dropped_numeric_cols=dropped_numeric_cols,
    dropped_nzv_cols=dropped_nzv_cols,
    feature_lists=feature_lists,
    output_path="../data/processed/feature_engineering_metadata.json",
)


Dataset feature-engineered disimpan ke: ../data/processed/application_train_featured.csv
Shape: (307511, 73)
Metadata Feature Engineering disimpan ke: ../data/processed/feature_engineering_metadata.json
